<a href="https://colab.research.google.com/github/abhipatidar343343/Logic_apps/blob/main/test_template_creator_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [448]:
import json
import os

# Define file paths for the reference JSON files
arm_schema_path = '/content/ARM_schema.json.json'
original_logic_app_path = '/content/original_logic_app.json.json'
parameterise_logic_app_path = '/content/parameterise_logic_app.json.json'

# Define the default path for the input Logic App JSON file
input_logic_app_path = '/content/input_logic_app.json.json'

print(f"Libraries 'json' and 'os' imported. File paths defined:")
print(f"ARM Schema: {arm_schema_path}")
print(f"Original Logic App: {original_logic_app_path}")
print(f"Parameterised Logic App: {parameterise_logic_app_path}")
print(f"Input Logic App: {input_logic_app_path}")

Libraries 'json' and 'os' imported. File paths defined:
ARM Schema: /content/ARM_schema.json.json
Original Logic App: /content/original_logic_app.json.json
Parameterised Logic App: /content/parameterise_logic_app.json.json
Input Logic App: /content/input_logic_app.json.json


In [449]:
def load_json_file(filepath):
    """Loads a JSON file and returns its content. Raises an error if the file is not found or invalid."""
    try:
        with open(filepath, 'r') as f:
            content = json.load(f)
        print(f"Successfully loaded and validated: {filepath}")
        return content
    except FileNotFoundError:
        raise FileNotFoundError(f"Error: The file '{filepath}' was not found.")
    except json.JSONDecodeError as e:
        raise ValueError(f"Error: The file '{filepath}' contains invalid JSON. Details: {e}")
    except Exception as e:
        raise Exception(f"An unexpected error occurred while loading '{filepath}': {e}")

# Load all four JSON files
try:
    arm_schema = load_json_file(arm_schema_path)
    original_logic_app = load_json_file(original_logic_app_path)
    parameterise_logic_app = load_json_file(parameterise_logic_app_path)
    input_logic_app = load_json_file(input_logic_app_path)

    print("\nAll JSON files loaded and validated successfully!")

    # Initialize an empty Python dictionary as a working object as per the main task
    working_object = {}
    print("Empty working object initialized.")

except (FileNotFoundError, ValueError, Exception) as e:
    print(f"Failed to load one or more JSON files: {e}")
    # Depending on requirements, you might want to exit or handle this error more gracefully
    # For this task, simply printing the error is sufficient for now.


Successfully loaded and validated: /content/ARM_schema.json.json
Successfully loaded and validated: /content/original_logic_app.json.json
Successfully loaded and validated: /content/parameterise_logic_app.json.json
Successfully loaded and validated: /content/input_logic_app.json.json

All JSON files loaded and validated successfully!
Empty working object initialized.


In [450]:
logic_app_workflow_resource = None

if 'resources' in input_logic_app and isinstance(input_logic_app['resources'], list):
    for resource in input_logic_app['resources']:
        if resource.get('type') == 'Microsoft.Logic/workflows':
            logic_app_workflow_resource = resource
            break

if logic_app_workflow_resource:
    print("Successfully identified and extracted the 'Microsoft.Logic/workflows' resource.")
    # print(json.dumps(logic_app_workflow_resource, indent=2)) # Uncomment to see the extracted resource
else:
    raise ValueError("Error: 'Microsoft.Logic/workflows' resource not found in the input Logic App JSON.")

Successfully identified and extracted the 'Microsoft.Logic/workflows' resource.


In [451]:
import json

# SECTION 2: Create/Normalize Top-Level Shell

# Ensure working_object is initialized as an empty dictionary if it's not already
if 'working_object' not in locals():
    working_object = {}
    print("Warning: 'working_object' was not found, initializing as empty dictionary.")

# Set $schema and contentVersion from ARM_schema
# Using .get() for safety, though ARM_schema is expected to have these
working_object['$schema'] = arm_schema.get('$schema', 'https://schema.management.azure.com/schemas/2019-04-01/deploymentTemplate.json#')
working_object['contentVersion'] = arm_schema.get('contentVersion', '1.0.0.0')

# Set metadata from ARM_schema
# If ARM_schema has metadata, copy it. Otherwise, initialize as empty or default.
working_object['metadata'] = arm_schema.get('metadata', {
    "title": "Logic app Title",
    "description": "Logic app description",
    "prerequisites": "",
    "lastUpdateTime": "",
    "author": {}
})

# Ensure parameters, variables, and resources exist as empty dictionaries or lists
if 'parameters' not in working_object:
    working_object['parameters'] = {}
if 'variables' not in working_object:
    working_object['variables'] = {}
if 'resources' not in working_object:
    working_object['resources'] = []

print("Section 2: Top-level shell created/normalized successfully.")
# print(json.dumps(working_object, indent=2)) # Uncomment to inspect the working_object after this section

Section 2: Top-level shell created/normalized successfully.


In [452]:
import re

print("Section X: Identifying canonical API names and types...")

# 1. Initialize an empty dictionary named canonical_api_names_map
canonical_api_names_map = {}

# 2. Initialize another empty dictionary named canonical_api_details
canonical_api_details = {}

# 3. Access the connections_value dictionary
connections_value = logic_app_workflow_resource.get('properties', {}).get('parameters', {}).get('$connections', {}).get('value', {})

if connections_value:
    for connection_key, connection_details in connections_value.items():
        # 5. Extract its full id string
        connection_id = connection_details.get('id', '')

        api_type = None
        raw_api_name = None

        # 6. Determine the api_type
        if '/managedApis/' in connection_id:
            api_type = 'managedApis'
            raw_api_name = connection_id.split('/managedApis/')[-1]
        elif '/customApis/' in connection_id:
            api_type = 'customApis'
            raw_api_name = connection_id.split('/customApis/')[-1]
        elif 'customApis_' in connection_id:
            # Handle cases where it's already a parameter reference like [parameters('customApis_..._externalid')]
            api_type = 'customApis'
            raw_api_name = connection_key
        elif 'connections_' in connection_id:
            api_type = 'managedApis'
            raw_api_name = connection_key
        else:
            print(f"  Warning: Could not determine API type for connection ID: {connection_id}. Skipping consolidation for {connection_key}.")
            continue

        if raw_api_name and api_type:
            # 8. Clean the raw API name by removing numerical suffixes
            canonical_api_name = re.sub(r'-\d+$', '', raw_api_name)

            # 9. Add an entry to canonical_api_names_map
            canonical_api_names_map[connection_key] = canonical_api_name

            # 10. Add to canonical_api_details if not already present
            if canonical_api_name not in canonical_api_details:
                canonical_api_details[canonical_api_name] = {'type': api_type}
                print(f"  Identified canonical API '{canonical_api_name}' of type '{api_type}'.")

print("Section X: Canonical API names and types identified successfully.")

Section X: Identifying canonical API names and types...
  Identified canonical API 'azuresentinel' of type 'managedApis'.
  Identified canonical API 'azuremonitorlogs' of type 'managedApis'.
  Identified canonical API 'ipapicustomconnector' of type 'customApis'.
  Identified canonical API 'abuseipdbapi' of type 'customApis'.
  Identified canonical API 'teams' of type 'managedApis'.
  Identified canonical API 'filesystem' of type 'managedApis'.
  Identified canonical API 'office365' of type 'managedApis'.
Section X: Canonical API names and types identified successfully.


In [453]:
import json

# SECTION 3: Build Parameters Section

# Helper function to add a parameter if it doesn't already exist
def add_parameter(param_name, param_type, default_value):
    # Modified: Removed 'description' and 'display_name' from function signature
    # to ensure no 'metadata' blocks are generated, as per the task.
    if param_name not in working_object['parameters']:
        param_entry = {
            "type": param_type,
            "defaultValue": default_value,
        }
        # No metadata block is added here, fulfilling the requirement.
        working_object['parameters'][param_name] = param_entry
        print(f"  Added parameter: {param_name}")
    else:
        print(f"  Parameter {param_name} already exists, skipping.")

# Helper function to check for and extract Log Analytics workspace references
def extract_workspace_references(logic_app_resource):
    definition = logic_app_resource.get('properties', {}).get('definition', {})
    triggers = definition.get('triggers', {})
    actions = definition.get('actions', {})

    # Default values for extraction
    extracted_subscription_id = None
    extracted_resource_group = None
    extracted_workspace_name = None
    is_workspace_referenced = False

    # Helper to process paths
    def process_path(path):
        nonlocal extracted_subscription_id, extracted_resource_group, extracted_workspace_name, is_workspace_referenced
        if "/subscriptions/" in path and "/resourceGroups/" in path and "/workspaces/" in path:
            is_workspace_referenced = True
            try:
                # Attempt to extract values from the path
                sub_start = path.find("/subscriptions/") + len("/subscriptions/")
                sub_end = path.find("/", sub_start)
                extracted_subscription_id = path[sub_start:sub_end]

                rg_start = path.find("/resourceGroups/") + len("/resourceGroups/")
                rg_end = path.find("/", rg_start)
                extracted_resource_group = path[rg_start:rg_end]

                ws_start = path.find("/workspaces/") + len("/workspaces/")
                ws_end = path.find("/", ws_start) if path.find("/", ws_start) != -1 else len(path)
                extracted_workspace_name = path[ws_start:ws_end]
            except Exception as e:
                print(f"  Warning: Could not fully extract workspace details from path: {path}. Error: {e}")

    # Check triggers
    for trigger_name, trigger_details in triggers.items():
        if trigger_details.get('type') == 'ApiConnection' and 'inputs' in trigger_details:
            process_path(trigger_details['inputs'].get('path', ''))
            if is_workspace_referenced: break # Stop if found in triggers

    # Check actions if not found in triggers
    if not is_workspace_referenced:
        for action_name, action_details in actions.items():
            if action_details.get('type') == 'ApiConnection' and 'inputs' in action_details:
                process_path(action_details['inputs'].get('path', ''))
                if is_workspace_referenced: break # Stop if found in actions

    return {
        'is_referenced': is_workspace_referenced,
        'subscriptionId': extracted_subscription_id,
        'resourceGroup': extracted_resource_group,
        'workspaceName': extracted_workspace_name
    }

print("Section 3: Building parameters section...")

# Resolve the default logic app name for use in defaultValues as a NEUTRAL placeholder
default_logic_app_name_resolved = "logic-app-name"

# 1. Parameterize Logic App Name with neutral default and standardize to 'PlaybookName'
logic_app_name_param_for_ref = "PlaybookName" # Instruction 1: Always set to 'PlaybookName'

add_parameter(
    param_name=logic_app_name_param_for_ref,
    param_type="String", # Instruction 2: param_type to 'String'
    default_value=default_logic_app_name_resolved # Instruction 2: No description, no display_name to suppress metadata
)

# 2. Conditionally add parameters for Workspace details
workspace_refs = extract_workspace_references(logic_app_workflow_resource)

if workspace_refs['is_referenced']:
    print("  Log Analytics workspace references detected. Adding workspace parameters.")

    add_parameter(
        param_name="workspaceSubscriptionId",
        param_type="String", # Instruction 3: param_type to 'String'
        default_value="your-subscription-id" # Instruction 3: No description, no display_name
    )

    add_parameter(
        param_name="workspaceResourceGroup",
        param_type="String", # Instruction 3: param_type to 'String'
        default_value="your-resource-group" # Instruction 3: No description, no display_name
    )

    add_parameter(
        param_name="workspaceName",
        param_type="String", # Instruction 3: param_type to 'String'
        default_value="your-log-analytics-workspace-name" # Instruction 3: No description, no display_name
    )
else:
    print("  No Log Analytics workspace references detected. Skipping workspace parameters.")

# 3. Parameterize Connection External IDs AND Connection Names using canonical names
connections_value = logic_app_workflow_resource.get('properties', {}).get('parameters', {}).get('$connections', {}).get('value', {})

if connections_value:
    print("  Processing connection parameters...")
    generated_canonical_params = set() # To ensure parameters are generated only once per canonical API

    for connection_key, connection_details in connections_value.items():
        canonical_api_name = canonical_api_names_map.get(connection_key)
        if not canonical_api_name:
            print(f"  Skipping connection {connection_key} as canonical name could not be determined.")
            continue

        api_type = canonical_api_details.get(canonical_api_name, {}).get('type')
        if not api_type:
            print(f"  Skipping connection {connection_key} as API type for canonical '{canonical_api_name}' could not be determined.")
            continue

        # Generate parameters only once per canonical API name
        if canonical_api_name in generated_canonical_params:
            continue # Parameters already added for this canonical API

        # Instruction 4a: Dynamically define default_ext_id_value
        default_ext_id_value = f"your-connection-id-{api_type}-{canonical_api_name}"
        # Parameter for Connection External ID (now based on canonical API name)
        param_ext_id_name = f"connections_{canonical_api_name}_externalid"

        # The external ID parameter is no longer used, as connectionId is handled via variables and resourceId in Section 11
        # Removing the add_parameter call for connections_*_externalid

        # Instruction 4b: Dynamically define default_conn_name_value
        default_conn_name_value = f"your-connection-name-{canonical_api_name}"
        # Parameter for Connection Name (now based on canonical API name)
        param_conn_name = f"connections_{canonical_api_name}_name"

        # The connection name parameter is no longer used, as connectionName is handled via variables in Section 11
        # Removing the add_parameter call for connections_*_name

        generated_canonical_params.add(canonical_api_name)
else:
    print("  No connections found to parameterize in 'properties.parameters.$connections.value'.")


print("Section 3: Parameters section built successfully.")
# print(json.dumps(working_object['parameters'], indent=2)) # Uncomment to inspect the parameters after this section

Section 3: Building parameters section...
  Added parameter: PlaybookName
  No Log Analytics workspace references detected. Skipping workspace parameters.
  Processing connection parameters...
Section 3: Parameters section built successfully.


In [454]:
import json

# SECTION 4: Build Variables Section

# Helper function to add a variable if it doesn't already exist
def add_variable(var_name, var_value):
    if var_name not in working_object['variables']:
        working_object['variables'][var_name] = var_value
        print(f"  Added variable: {var_name}")
    else:
        print(f"  Variable {var_name} already exists, skipping.")

print("Section 4: Building variables section...")

# 1. Define logicAppName variable if the logic app name was parameterized
# The parameter name for the Logic App is now standardized to 'PlaybookName'
logic_app_name_param_for_variable = "PlaybookName"

# Add logicAppName variable using the standardized PlaybookName parameter
add_variable(
    var_name="logicAppName",
    var_value=f"[parameters('{logic_app_name_param_for_variable}')]"
)

# 2. Add 'location' variable to reference the 'location' parameter
# Removed as per user request.
# add_variable(
#     var_name="location",
#     var_value="[parameters('location')]"
# )

# 3. Parameterize Connection Names for use as variables using canonical names
if canonical_api_details:
    print("  Processing connection name variables using canonical API names...")
    for canonical_api_name in canonical_api_details.keys():
        var_name = f"{canonical_api_name}ConnectionName"
        # Construct the variable value to follow the pattern [concat('<apiName>-', parameters('PlaybookName'))]
        var_value = f"[concat('{canonical_api_name}-', parameters('{logic_app_name_param_for_variable}'))]"
        add_variable(var_name, var_value)
else:
    print("  No canonical API details found to build connection name variables for.")

print("Section 4: Variables section built successfully.")
# print(json.dumps(working_object['variables'], indent=2)) # Uncomment to inspect the variables after this section

Section 4: Building variables section...
  Added variable: logicAppName
  Processing connection name variables using canonical API names...
  Added variable: azuresentinelConnectionName
  Added variable: azuremonitorlogsConnectionName
  Added variable: ipapicustomconnectorConnectionName
  Added variable: abuseipdbapiConnectionName
  Added variable: teamsConnectionName
  Added variable: filesystemConnectionName
  Added variable: office365ConnectionName
Section 4: Variables section built successfully.


In [455]:
import json

# SECTION 5: Build Microsoft.Web/connections resources

print("Section 5: Building Microsoft.Web/connections resources...")

# Clear existing resources before rebuilding
working_object['resources'] = []

if canonical_api_details:
    for canonical_api_name, details in canonical_api_details.items():
        api_type = details.get('type')

        if not api_type:
            print(f"  Warning: No API type found for canonical API '{canonical_api_name}'. Skipping.")
            continue

        connection_name_reference = f"[variables('{canonical_api_name}ConnectionName')]"

        # Dynamically build the API ID based on whether it is a managed or custom API
        # Managed APIs use the location-based path
        if api_type == 'managedApis':
            api_id_expression = f"[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/{canonical_api_name}')]"
        # Custom APIs use the resource group-based path (no location segment required in the ID string)
        else:
            api_id_expression = f"[concat('/subscriptions/', subscription().subscriptionId, '/resourceGroups/', resourceGroup().name, '/providers/Microsoft.Web/customApis/{canonical_api_name}')]"

        connection_resource = {
            "type": "Microsoft.Web/connections",
            "apiVersion": "2016-06-01",
            "name": connection_name_reference,
            "location": "[resourceGroup().location]",
            "properties": {
                "api": {
                    "id": api_id_expression
                },
                "displayName": canonical_api_name.title()
            }
        }

        working_object['resources'].append(connection_resource)
        print(f"  Added consolidated connection resource for: {canonical_api_name} ({api_type})")

else:
    print("  No canonical API details found to build Microsoft.Web/connections resources for.")

print("Section 5: Microsoft.Web/connections resources built successfully with corrected API IDs.")

Section 5: Building Microsoft.Web/connections resources...
  Added consolidated connection resource for: azuresentinel (managedApis)
  Added consolidated connection resource for: azuremonitorlogs (managedApis)
  Added consolidated connection resource for: ipapicustomconnector (customApis)
  Added consolidated connection resource for: abuseipdbapi (customApis)
  Added consolidated connection resource for: teams (managedApis)
  Added consolidated connection resource for: filesystem (managedApis)
  Added consolidated connection resource for: office365 (managedApis)
Section 5: Microsoft.Web/connections resources built successfully with corrected API IDs.


In [456]:
import json

# SECTION 6: Build Microsoft.Logic/workflows resource shell

print("Section 6: Building Microsoft.Logic/workflows resource shell...")

# Prepare the dependsOn array
depends_on_connections = []
for resource in working_object['resources']:
    if resource.get('type') == 'Microsoft.Web/connections':
        # Extract the connection name expression, e.g., "[variables('azuresentinelConnectionName')]"
        connection_name_expr = resource.get('name')
        if connection_name_expr:
            # Construct the full resourceId expression for the dependency
            # Corrected: Strip the outer square brackets from connection_name_expr
            depends_on_connections.append(f"[resourceId('Microsoft.Web/connections', {connection_name_expr.strip('[]')})]")

# Get logic app name from parameters
# The logic app name parameter is now standardized to 'PlaybookName'
logic_app_name_param_value = "[parameters('PlaybookName')]"

# Build the main Microsoft.Logic/workflows resource
logic_app_resource = {
    "type": "Microsoft.Logic/workflows",
    "apiVersion": logic_app_workflow_resource.get('apiVersion', '2017-07-01'), # Default API version if not present
    "name": logic_app_name_param_value,
    "location": "[resourceGroup().location]", # Using resource group location directly
    "identity": logic_app_workflow_resource.get('identity', {'type': 'SystemAssigned'}), # Keep existing identity or default
    "properties": {}, # Initialize properties as an empty dictionary
    "dependsOn": depends_on_connections
}

# Remove any existing Microsoft.Logic/workflows resource before adding the new one
working_object['resources'] = [res for res in working_object['resources'] if res.get('type') != 'Microsoft.Logic/workflows']

# Add the logic app resource to the working object's resources
working_object['resources'].append(logic_app_resource)
print("Section 6: Microsoft.Logic/workflows resource shell built successfully.")
# print(json.dumps(working_object['resources'], indent=2)) # Uncomment to inspect the resources after this section

Section 6: Building Microsoft.Logic/workflows resource shell...
Section 6: Microsoft.Logic/workflows resource shell built successfully.


In [457]:
import json

# SECTION 7: Build workflow definition.parameters

print("Section 7: Building workflow definition.parameters...")

# Helper function to ensure the Microsoft.Logic/workflows resource is in working_object['resources']
def _ensure_logic_app_workflow_resource(working_object, logic_app_workflow_resource):
    logic_app_res = None
    for resource in working_object['resources']:
        if resource.get('type') == 'Microsoft.Logic/workflows':
            logic_app_res = resource
            break

    if logic_app_res:
        return logic_app_res
    else:
        print("  Microsoft.Logic/workflows resource not found in working_object. Rebuilding it now (from Section 6 logic).")
        # Re-run Section 6 logic to build and add the resource
        depends_on_connections = []
        for resource in working_object['resources']: # Now working_object['resources'] only has Microsoft.Web/connections
            if resource.get('type') == 'Microsoft.Web/connections':
                connection_name_expr = resource.get('name')
                if connection_name_expr:
                    depends_on_connections.append(f"[resourceId('Microsoft.Web/connections', {connection_name_expr.strip('[]')})]")

        logic_app_name_param_value = None
        for param_name, param_details in working_object.get('parameters', {}).items():
            if param_details.get('metadata', {}).get('displayName') == 'Logic App Name':
                logic_app_name_param_value = f"[parameters('{param_name}')]"
                break
        # Fallback if logic app name parameter not found, use original resource name or a generic parameter
        if not logic_app_name_param_value:
            logic_app_name_param_value = logic_app_workflow_resource.get('name', "[parameters('logicAppName')]")

        # Build the main Microsoft.Logic/workflows resource
        logic_app_res = {
            "type": "Microsoft.Logic/workflows",
            "apiVersion": logic_app_workflow_resource.get('apiVersion', '2017-07-01'), # Default API version if not present
            "name": logic_app_name_param_value,
            "location": "[resourceGroup().location]", # Using resource group location directly
            "identity": logic_app_workflow_resource.get('identity', {'type': 'SystemAssigned'}), # Keep existing identity or default
            "properties": {}, # Initialize properties as an empty dictionary
            "dependsOn": depends_on_connections
        }

        working_object['resources'].append(logic_app_res)
        print("  Microsoft.Logic/workflows resource rebuilt and added.")
        return logic_app_res


# Find or rebuild the Microsoft.Logic/workflows resource in the working_object
logic_app_resource_in_working_object = _ensure_logic_app_workflow_resource(working_object, logic_app_workflow_resource)

if not logic_app_resource_in_working_object:
    raise ValueError("Section 7 Error: Microsoft.Logic/workflows resource not found in working_object.")

# Ensure properties key exists
if 'properties' not in logic_app_resource_in_working_object:
    logic_app_resource_in_working_object['properties'] = {}

# Set the definition with required schema and contentVersion
definition = {
    "$schema": "https://schema.management.azure.com/providers/Microsoft.Logic/schemas/2016-06-01/workflowdefinition.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {}
}
logic_app_resource_in_working_object['properties']['definition'] = definition

# Extract original parameters from the input logic app's definition
original_definition_parameters = logic_app_workflow_resource.get('properties', {}).get('definition', {}).get('parameters', {})

new_definition_parameters = {}

for param_name, param_details in original_definition_parameters.items():
    new_param_details = param_details.copy()

    # Handle $connections parameters within the definition
    if param_name.startswith('$connections') and isinstance(new_param_details.get('defaultValue'), dict):
        default_value = new_param_details['defaultValue'].get('value', {})
        if isinstance(default_value, dict) and 'connection' in default_value:
            connection_id = default_value['connection'].get('id')
            if connection_id:
                raw_api_name_match = None
                if '/managedApis/' in connection_id:
                    raw_api_name_match = connection_id.split('/managedApis/')[-1]
                elif '/customApis/' in connection_id:
                    raw_api_name_match = connection_id.split('/customApis/')[-1]

                if raw_api_name_match:
                    canonical_api_name = canonical_api_names_map.get(raw_api_name_match)
                    if canonical_api_name:
                        connection_variable_name = f"{canonical_api_name}ConnectionName"
                        new_param_details['defaultValue']['value']['connection']['id'] = f"[resourceId('Microsoft.Web/connections', variables('{connection_variable_name}'))]"
                        print(f"  Parameterized internal connection ID for '{param_name}' using canonical '{canonical_api_name}'.")

    new_definition_parameters[param_name] = new_param_details

# Assign the new parameters to the workflow definition
logic_app_resource_in_working_object['properties']['definition']['parameters'] = new_definition_parameters

print("Section 7: Workflow definition built with $schema, contentVersion and parameters successfully.")

Section 7: Building workflow definition.parameters...
Section 7: Workflow definition built with $schema, contentVersion and parameters successfully.


In [458]:
import json
import re

# SECTION 8: Build workflow definition.triggers

print("Section 8: Building workflow definition.triggers with fixed connection names...")

logic_app_resource_in_working_object = next((res for res in working_object['resources'] if res.get('type') == 'Microsoft.Logic/workflows'), None)

original_triggers = logic_app_workflow_resource.get('properties', {}).get('definition', {}).get('triggers', {})
new_triggers_str = json.dumps(original_triggers)

# 1. Standardize PlaybookName parameter reference
old_param_pattern = r"parameters\('workflows_AEAZ_Firewall_IP_Blocking_name'\)"
new_triggers_str = re.sub(old_param_pattern, "parameters('PlaybookName')", new_triggers_str)

# 2. Standardize Connection names within inputs.host.connection.name
# Find patterns like @parameters('$connections')['azuresentinel']['connectionId']
# and map them to variables('azuresentinelConnectionName')
def replace_connection_expr(match):
    conn_key = match.group(1)
    # Check if we have a canonical mapping for this key
    canonical_name = canonical_api_names_map.get(conn_key, conn_key)
    return f"@parameters('$connections')['{conn_key}']['connectionId']"

# Actually, for triggers and actions to work with the consolidated parameter structure,
# the host.connection.name should reference the key in the workflow's parameters.$connections
# No change is strictly needed here if Section 11 is consistent with the trigger keys.
# However, the error 'azuremonitorlogs-test_5_deployemnt_template' suggests a mismatch in Section 11 naming.

new_triggers = json.loads(new_triggers_str)

# Apply path replacements for Workspace
replacements = {
    "/subscriptions/05482007-a538-44e4-8e37-7cab6b74255e": "/subscriptions/@{encodeURIComponent(parameters('workspaceSubscriptionId'))}",
    "/resourceGroups/AE_AZURE_SE_RG": "/resourceGroups/@{encodeURIComponent(parameters('workspaceResourceGroup'))}",
    "/workspaces/AE-AZ-AzureSentinel": "/workspaces/@{encodeURIComponent(parameters('workspaceName'))}"
}

for trigger_name, trigger_details in new_triggers.items():
    if trigger_details.get('type') == 'ApiConnection' or trigger_details.get('type') == 'ApiConnectionWebhook':
        if 'inputs' in trigger_details and 'path' in trigger_details['inputs']:
            modified_path = trigger_details['inputs']['path']
            for old_value, new_value in replacements.items():
                modified_path = modified_path.replace(old_value, new_value)
            trigger_details['inputs']['path'] = modified_path

logic_app_resource_in_working_object['properties']['definition']['triggers'] = new_triggers
print("Section 8: Triggers built successfully.")

Section 8: Building workflow definition.triggers with fixed connection names...
Section 8: Triggers built successfully.


In [459]:
import json
import re

# SECTION 9: Build workflow definition.actions

print("Section 9: Building workflow definition.actions with fixed connection names...")

logic_app_resource_in_working_object = next((res for res in working_object['resources'] if res.get('type') == 'Microsoft.Logic/workflows'), None)

original_actions = logic_app_workflow_resource.get('properties', {}).get('definition', {}).get('actions', {})
new_actions_str = json.dumps(original_actions)

# 1. Standardize PlaybookName parameter reference
old_param_pattern = r"parameters\('workflows_AEAZ_Firewall_IP_Blocking_name'\)"
new_actions_str = re.sub(old_param_pattern, "parameters('PlaybookName')", new_actions_str)

new_actions = json.loads(new_actions_str)

# Apply path replacements for Workspace
replacements = {
    "/subscriptions/05482007-a538-44e4-8e37-7cab6b74255e": "/subscriptions/@{encodeURIComponent(parameters('workspaceSubscriptionId'))}",
    "/resourceGroups/AE_AZURE_SE_RG": "/resourceGroups/@{encodeURIComponent(parameters('workspaceResourceGroup'))}",
    "/workspaces/AE-AZ-AzureSentinel": "/workspaces/@{encodeURIComponent(parameters('workspaceName'))}"
}

for action_name, action_details in new_actions.items():
    if action_details.get('type') == 'ApiConnection':
        if 'inputs' in action_details and 'path' in action_details['inputs']:
            modified_path = action_details['inputs']['path']
            for old_value, new_value in replacements.items():
                modified_path = modified_path.replace(old_value, new_value)
            action_details['inputs']['path'] = modified_path

logic_app_resource_in_working_object['properties']['definition']['actions'] = new_actions
print("Section 9: Actions built successfully.")

Section 9: Building workflow definition.actions with fixed connection names...
Section 9: Actions built successfully.


In [460]:
import json

# SECTION 10: Build workflow definition.outputs

print("Section 10: Building workflow definition.outputs...")

# Find the Microsoft.Logic/workflows resource in the working_object
logic_app_resource_in_working_object = None
for resource in working_object['resources']:
    if resource.get('type') == 'Microsoft.Logic/workflows':
        logic_app_resource_in_working_object = resource
        break

if not logic_app_resource_in_working_object:
    raise ValueError("Section 10 Error: Microsoft.Logic/workflows resource not found in working_object.")

# Ensure properties and definition keys exist
if 'properties' not in logic_app_resource_in_working_object:
    logic_app_resource_in_working_object['properties'] = {}
if 'definition' not in logic_app_resource_in_working_object['properties']:
    logic_app_resource_in_working_object['properties']['definition'] = {}

# Extract original outputs from the input logic app's definition
original_outputs = logic_app_workflow_resource.get('properties', {}).get('definition', {}).get('outputs', {})

# Assign the outputs to the workflow definition
logic_app_resource_in_working_object['properties']['definition']['outputs'] = original_outputs

print("Section 10: Workflow definition.outputs built successfully.")
# print(json.dumps(logic_app_resource_in_working_object['properties']['definition']['outputs'], indent=2)) # Uncomment to inspect


Section 10: Building workflow definition.outputs...
Section 10: Workflow definition.outputs built successfully.


In [461]:
import json

# SECTION 11: Build properties.parameters.$connections.value

print("Section 11: Building properties.parameters.$connections.value...")

# Helper function to ensure the Microsoft.Logic/workflows resource is in working_object['resources']
def _ensure_logic_app_workflow_resource(working_object, logic_app_workflow_resource):
    logic_app_res = None
    for resource in working_object['resources']:
        if resource.get('type') == 'Microsoft.Logic/workflows':
            logic_app_res = resource
            break

    if logic_app_res:
        return logic_app_res
    else:
        print("  Rebuilding Logic App workflow resource shell...")
        depends_on_connections = []
        for resource in working_object['resources']:
            if resource.get('type') == 'Microsoft.Web/connections':
                name_expr = resource.get('name')
                if name_expr:
                    depends_on_connections.append(f"[resourceId('Microsoft.Web/connections', {name_expr.strip('[]')})]")

        logic_app_res = {
            "type": "Microsoft.Logic/workflows",
            "apiVersion": "2017-07-01",
            "name": "[parameters('PlaybookName')]",
            "location": "[resourceGroup().location]",
            "identity": {"type": "SystemAssigned"},
            "properties": {},
            "dependsOn": depends_on_connections
        }
        working_object['resources'].append(logic_app_res)
        return logic_app_res

logic_app_resource_in_working_object = _ensure_logic_app_workflow_resource(working_object, logic_app_workflow_resource)

if 'properties' not in logic_app_resource_in_working_object:
    logic_app_resource_in_working_object['properties'] = {}
if 'parameters' not in logic_app_resource_in_working_object['properties']:
    logic_app_resource_in_working_object['properties']['parameters'] = {}
if '$connections' not in logic_app_resource_in_working_object['properties']['parameters']:
    logic_app_resource_in_working_object['properties']['parameters']['$connections'] = {}

original_connections_value = logic_app_workflow_resource.get('properties', {}).get('parameters', {}).get('$connections', {}).get('value', {})
new_connections_value = json.loads(json.dumps(original_connections_value))

connection_key_to_canonical_map = {k: v for k, v in canonical_api_names_map.items()}

for connection_key in list(new_connections_value.keys()):
    canonical_api_name = connection_key_to_canonical_map.get(connection_key)
    if not canonical_api_name:
        new_connections_value.pop(connection_key, None)
        continue

    api_type = canonical_api_details.get(canonical_api_name, {}).get('type')
    if not api_type:
        new_connections_value.pop(connection_key, None)
        continue

    # Update the internal 'id' using the correct API type logic
    if api_type == 'managedApis':
        new_connections_value[connection_key]['id'] = f"[concat('/subscriptions/', subscription().subscriptionId, '/providers/Microsoft.Web/locations/', resourceGroup().location, '/managedApis/{canonical_api_name}')]"
    else:
        # For customApis, we use the connection_key or the part after /customApis/ if it existed
        orig_id = original_connections_value.get(connection_key, {}).get('id', '')
        api_name_for_id = canonical_api_name
        if '/customApis/' in orig_id:
            api_name_for_id = orig_id.split('/customApis/')[-1]

        new_connections_value[connection_key]['id'] = f"[concat('/subscriptions/', subscription().subscriptionId, '/resourceGroups/', resourceGroup().name, '/providers/Microsoft.Web/customApis/{api_name_for_id}')]"

    connection_variable_name = f"{canonical_api_name}ConnectionName"
    new_connections_value[connection_key]['connectionId'] = f"[resourceId('Microsoft.Web/connections', variables('{connection_variable_name}'))]"
    new_connections_value[connection_key]['connectionName'] = f"[variables('{connection_variable_name}')]"

    if 'connectionProperties' in new_connections_value[connection_key]:
        del new_connections_value[connection_key]['connectionProperties']

logic_app_resource_in_working_object['properties']['parameters']['$connections']['value'] = new_connections_value
print("Section 11: Updated with correct API types for internal connection mappings (Custom API fix).")

Section 11: Building properties.parameters.$connections.value...
Section 11: Updated with correct API types for internal connection mappings (Custom API fix).


In [462]:
import json
import os

# SECTION 12: Final Validation and Saving Output

print("Section 12: Performing final validation and saving output...")

# --- Final Validation (Basic Check) ---
validation_errors = []
if not working_object:
    validation_errors.append("Working object is empty.")
else:
    required_keys = ['$schema', 'contentVersion', 'parameters', 'variables', 'resources']
    for key in required_keys:
        if key not in working_object:
            validation_errors.append(f"Missing top-level key: '{key}'.")
    if not isinstance(working_object.get('resources'), list) or not working_object['resources']:
        validation_errors.append("Resources section is missing or empty.")

    # Check for the main Logic App resource
    logic_app_resource_found = False
    for res in working_object.get('resources', []):
        if res.get('type') == 'Microsoft.Logic/workflows':
            logic_app_resource_found = True
            break
    if not logic_app_resource_found:
        validation_errors.append("Main 'Microsoft.Logic/workflows' resource not found in output.")

if validation_errors:
    print("Final Validation Warnings/Errors:")
    for error in validation_errors:
        print(f"  - {error}")
    print("Output may be malformed. Saving anyway...")
else:
    print("Basic validation passed: working_object appears well-formed.")

# --- Save Output JSON File ---
output_file_path = '/content/output_logic_app.json'

try:
    with open(output_file_path, 'w') as f:
        json.dump(working_object, f, indent=2)
    print(f"Output successfully saved to: {output_file_path}")
except Exception as e:
    print(f"Error saving output file '{output_file_path}': {e}")

print("Section 12 complete. All sections are finished.")

Section 12: Performing final validation and saving output...
Basic validation passed: working_object appears well-formed.
Output successfully saved to: /content/output_logic_app.json
Section 12 complete. All sections are finished.
